In [1]:
import pandas as pd
import numpy as np

In [2]:
VMT_FILE = "VMT_2022v1_full_annual_20240226_monthly_18jun2024_nf_v4.csv"
OUTPUT_FILE = "VMT_2022v1_full_annual_20240226_monthly_18jun2024_nf_v4_ZeroPre2010s.csv"

CMAP_FIPS = ["17031", "17043", "17089", "17093", "17097", "17111", "17197"]
TRUCK_SOURCE_TYPES = [52, 53, 61, 62]

activity_columns = ["country_cd", "region_cd", "tribal_code", "census_tract_cd", "shape_id", "scc", "CD", "MSR", "activity_type", "ann_parm_value", "calc_year", "date_updated", "data_set_id", "jan_value", "feb_value", "mar_value", "apr_value", "may_value", "jun_value", "jul_value", "aug_value", "sep_value", "oct_value", "nov_value", "dec_value", "comment"]

# frac come from %VMTs previously calc using MOVESv4 Doc 
# 
legacy_vmt_fraction = {52: 0.2193, 53: 0.1955, 61: 0.1592, 62: 0.2605}

In [4]:
header_lines = []
with open(VMT_FILE, "r") as f:
    for line in f:
        if line.startswith("#"):
            header_lines.append(line)
        else:
            break
vmt = pd.read_csv(VMT_FILE, comment="#", header=None, names=activity_columns, dtype={"region_cd": str, "scc": str})

vmt["source_type"] = vmt["scc"].str[4:6].astype(int)

vmt_original = vmt.copy()

value_columns = ["ann_parm_value", "jan_value", "feb_value", "mar_value", "apr_value", "may_value", "jun_value", "jul_value", "aug_value", "sep_value", "oct_value", "nov_value", "dec_value"]

for source_type in TRUCK_SOURCE_TYPES:
    keep_fraction = 1.0 - legacy_vmt_fraction[source_type]
    mask = vmt["region_cd"].isin(CMAP_FIPS) & vmt["source_type"].eq(source_type)
    vmt.loc[mask, value_columns] = vmt.loc[mask, value_columns] * keep_fraction

before = vmt_original[vmt_original["region_cd"].isin(CMAP_FIPS) & vmt_original["source_type"].isin(TRUCK_SOURCE_TYPES)].groupby("source_type")["ann_parm_value"].sum()

after = vmt[vmt["region_cd"].isin(CMAP_FIPS) & vmt["source_type"].isin(TRUCK_SOURCE_TYPES)].groupby("source_type")["ann_parm_value"].sum()

validation = pd.DataFrame({"Original VMT": before, "ZeroPre2010s VMT": after})

validation["Legacy VMT Removed"] = validation["Original VMT"] - validation["ZeroPre2010s VMT"]

validation["Removed %"] = validation["Legacy VMT Removed"] / validation["Original VMT"] * 100

print("\n" + "=" * 75)
print("ZERO-PRE-2010 VMT VALIDATION")
print("=" * 75)            
print(validation.to_string(formatters={"Original VMT": lambda x: f"{x:,.0f}", "ZeroPre2010s VMT": lambda x: f"{x:,.0f}", "Legacy VMT Removed": lambda x: f"{x:,.0f}", "Removed %": lambda x: f"{x:.2f}%"}))

original_cmap_total = vmt_original[vmt_original["region_cd"].isin(CMAP_FIPS)]["ann_parm_value"].sum()

new_cmap_total = vmt[vmt["region_cd"].isin(CMAP_FIPS)]["ann_parm_value"].sum()

removed_cmap_total = original_cmap_total - new_cmap_total

print("\n" + "=" * 75)
print("TOTAL CMAP VMT")
print("=" * 75)

print(f"\nOriginal CMAP VMT: {original_cmap_total:,.0f} VMT/yr")
print(f"ZeroPre2010s CMAP VMT: {new_cmap_total:,.0f} VMT/yr")
print(f"Legacy truck VMT removed: {removed_cmap_total:,.0f} VMT/yr")
print(f"Share of total CMAP VMT removed: {removed_cmap_total / original_cmap_total * 100:.2f}%")



ZERO-PRE-2010 VMT VALIDATION
             Original VMT ZeroPre2010s VMT Legacy VMT Removed Removed %
source_type                                                            
52          1,712,494,640    1,336,944,566        375,550,075    21.93%
53            117,131,148       94,232,008         22,899,139    19.55%
61            785,377,278      660,345,215        125,032,063    15.92%
62          2,817,972,493    2,083,890,658        734,081,834    26.05%

TOTAL CMAP VMT

Original CMAP VMT: 59,446,334,969 VMT/yr
ZeroPre2010s CMAP VMT: 58,188,771,858 VMT/yr
Legacy truck VMT removed: 1,257,563,111 VMT/yr
Share of total CMAP VMT removed: 2.12%


In [5]:
vmt_output = vmt.drop(columns="source_type")

vmt_output = vmt_output[activity_columns]

with open(OUTPUT_FILE, "w") as f:
    for line in header_lines:
        f.write(line)

vmt_output.to_csv(OUTPUT_FILE, mode="a", index=False, header=False)

print("\n" + "=" * 75)
print("OUTPUT FILE CREATED")
print("=" * 75)
print(f"\n{OUTPUT_FILE}")


OUTPUT FILE CREATED

VMT_2022v1_full_annual_20240226_monthly_18jun2024_nf_v4_ZeroPre2010s.csv
